# 实验报告03：正则表达式与网络结构化数据
## ⼀、实验环境与提交要求
1. 使⽤独⽴的 conda 虚拟环境，不使⽤ base 环境。
2. 在第⼀个代码单元填写⾃⼰的学号，再按顺序运⾏各代码单元。可在 PyCharm 或
VS Code 中编辑 Notebook。
3. ⽤ Markdown 单元格解释匹配规则和解析结果（如果必要），⽤代码单元格完成计
算。每项任务应显⽰关键输出。
4. 最终只提交 学号_姓名_exp03.ipynb。
5. 提交前重启内核并运⾏全部代码，确认没有依赖旧变量。


## 1.1 第⼀个代码框：⾝份与环境记录

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080301

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]   {sys.version.split()[0]}")

[Conda 环境] exp02
[学号]       20244080301
[机器指纹] 4893301fd6adb35f
[RUN_ID]   20244080301-4893301fd6adb35f
[Python]   3.12.15


## 实验1：正则表达式的基础匹配

在本实验开始时导入re。操作任务：


### 1. 在"cat on mat; later: cats and dogs"上使用正则表达式，分别匹配文本中的数字\d+、\s+和\bcat\b

In [2]:
import re

text = "cat on mat; later: cats and dogs"
print("文本:", text)

print(r"\d+    匹配结果:", re.findall(r"\d+", text))
print(r"\s+    匹配结果:", re.findall(r"\s+", text))
print(r"\bcat\b 匹配结果:", re.findall(r"\bcat\b", text))

文本: cat on mat; later: cats and dogs
\d+    匹配结果: []
\s+    匹配结果: [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b 匹配结果: ['cat']


### 2. 在 "cat dog bird CAT" 上使⽤ cat|dog。⽤ re.findall(r"a|aa|aaa","aaaa") 和 re.findall(r"aaa|aa|a", "aaaa") 展⽰差异。


In [ ]:
text2 = "cat dog bird CAT"
print("文本:", text2)

print(r"cat|dog（区分大小写）:", re.findall(r"cat|dog", text2))
print(r"cat|dog（不区分大小写）:", re.findall(r"cat|dog", text2, re.IGNORECASE))

s = "aaaa"
print()
print(r"a|aa|aaa 对 aaaa:", re.findall(r"a|aa|aaa", s))
print(r"aaa|aa|a 对 aaaa:", re.findall(r"aaa|aa|a", s))

文本: cat dog bird CAT
cat|dog（区分大小写）: ['cat', 'dog']
cat|dog（忽略大小写）: ['cat', 'dog', 'CAT']

a|aa|aaa 对 aaaa: ['a', 'a', 'a', 'a']
aaa|aa|a 对 aaaa: ['aaa', 'a']


### 3. 在 "ct cat caat caaat" 上分别使⽤ ca*t、ca+t、ca?t、ca{2}t 和ca{1,2}t，输出匹配列表并说明各量词的含义。


In [4]:
text3 = "ct cat caat caaat"
print("文本:", text3)

patterns = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]
for pattern in patterns:
    print(f"{pattern:10s} -> {re.findall(pattern, text3)}")

文本: ct cat caat caaat
ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


## 实验2：分组、回溯引用与替换
操作任务：

### 1. 在 "联系：alice@example.com；备选：bob@test.org" 中，⽤捕获组提取每个邮箱地址的⽤⼾名和域名。


In [5]:
text4 = "联系：alice@example.com；备选：bob@test.org"
print("文本:", text4)

pattern = r"([\w.+-]+)@([\w-]+\.[\w.]+)"
matches = re.findall(pattern, text4)
print("捕获组结果:", matches)

for user, domain in matches:
    print(f"用户名: {user:8s} 域名: {domain}")

文本: 联系：alice@example.com；备选：bob@test.org
捕获组结果: [('alice', 'example.com'), ('bob', 'test.org')]
用户名: alice    域名: example.com
用户名: bob      域名: test.org


### 2. 在 "cat cat, dog cat, bird bird" 中，⽤后向引⽤找出连续重复的单词。

In [6]:
text5 = "cat cat, dog cat, bird bird"
print("文本:", text5)

print("重复出现的单词:", re.findall(r"\b(\w+)\s+\1\b", text5))

for m in re.finditer(r"\b(\w+)\s+\1\b", text5):
    print(f"命中片段: {m.group(0)!r}，重复的词: {m.group(1)}")

文本: cat cat, dog cat, bird bird
重复出现的单词: ['cat', 'bird']
命中片段: 'cat cat'，重复的词: cat
命中片段: 'bird bird'，重复的词: bird


### 3. 将 "2024-05-01\n2024/05/02\n2024.05.03" 中的⽇期从“年-⽉-⽇”形式转换为“⽉-⽇-年”形式，保留原来的⾏数。可⽤ re.sub()、捕获组和替换表达式完成；展⽰转换前后⽂本。

In [7]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"

print("转换前:")
print(dates)

converted = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", dates)

print("转换后:")
print(converted)
print("行数是否保持一致:", dates.count("\n") == converted.count("\n"))

转换前:
2024-05-01
2024/05/02
2024.05.03
转换后:
05-01-2024
05-02-2024
05-03-2024
行数是否保持一致: True


### 4. 将 "联系我：123-456-7890 或 987.654.3210" 中的电话号码统⼀为 (123)456-7890 形式。


In [8]:
text6 = "联系我：123-456-7890 或 987.654.3210"
print("原文:", text6)

unified = re.sub(r"(\d{3})[.-](\d{3})[.-](\d{4})", r"(\1) \2-\3", text6)
print("统一后:", unified)

原文: 联系我：123-456-7890 或 987.654.3210
统一后: 联系我：(123) 456-7890 或 (987) 654-3210


## 实验3：使用 BeautifulSoup 解析 HTML

使⽤以下字符串，不依赖⽹⻚下载：


### 1. ⽤ BeautifulSoup(html_doc, "html.parser") 构建解析树，输出⻚⾯标题的⽂本、h1 的标签名与⽂本。


In [9]:
from bs4 import BeautifulSoup

html_doc = '''<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>'''

soup = BeautifulSoup(html_doc, "html.parser")

print("页面标题:", soup.title.get_text())

h1 = soup.find("h1")
print("h1 标签名:", h1.name)
print("h1 文本:", h1.get_text())

页面标题: 实验课程
h1 标签名: h1
h1 文本: 数据分析


### 2. 使⽤ find_all() 找出所有 class="course" 的段落和所有链接；逐项输出链接⽂字与 href 属性。

In [10]:
paragraphs = soup.find_all("p", class_="course")
print(f"class='course' 的段落数: {len(paragraphs)}")
for p in paragraphs:
    print("  段落文本:", p.get_text())

links = soup.find_all("a")
print(f"链接数: {len(links)}")
for a in links:
    print(f"  链接文字: {a.get_text()} | href 属性: {a['href']}")

class='course' 的段落数: 2
  段落文本: 第六讲：正则表达式
  段落文本: 第七讲：结构化数据
链接数: 2
  链接文字: 讲义六 | href 属性: https://example.org/lecture06
  链接文字: 讲义七 | href 属性: https://example.org/lecture07


### 3. ⽤ get_text() 提取⻚⾯可⻅⽂本；⽐较整个⽂档、单个段落和单个链接的⽂本结果。

In [11]:
print("整个文档 get_text():")
print(repr(soup.get_text()))

print("单个段落 get_text():", repr(paragraphs[0].get_text()))
print("单个链接 get_text():", repr(links[0].get_text()))

print()
print("区别总结：整篇文本包含标题、h1、两段正文和两个链接文字；")
print("单个段落只含该段文字；单个链接只含链接文字，href 不进入文本。")

整个文档 get_text():
'实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n'
单个段落 get_text(): '第六讲：正则表达式'
单个链接 get_text(): '讲义六'

区别总结：整篇文本包含标题、h1、两段正文和两个链接文字；
单个段落只含该段文字；单个链接只含链接文字，href 不进入文本。
